In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

TARGET = "st_xfer_time_ms"

FEATURE_COLS = [
    "st_files",
    "st_dirs",
    "st_successful",
    "st_failed",
    "st_expired",
    "st_canceled",
    "st_bytes_xfered",
    "st_faults",
    "st_files_skipped",
    "st_skipped_errors",
]

DROP_COLS = ["grp_uuid", "user_id", "request_time", "complete_time", "src_host_ep_id", "dst_host_ep_id", "grp_delete", "grp_status"]

def safe_spearman(y_true, y_pred):
    corr = spearmanr(y_true, y_pred).correlation
    if corr is None or np.isnan(corr):
        return 0.0
    return corr


def pinball_loss(y_true, y_pred, tau):
    diff = y_true - y_pred
    return np.mean(np.maximum(tau * diff, (tau - 1) * diff))

def evaluate_model(model, X_tr, y_tr, X_te, y_te, y_te_raw):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    return {
        "mae_log": mean_absolute_error(y_te, pred),
        "spearman": safe_spearman(y_te, pred),
        "mae_log_top90": mean_absolute_error(
            y_te[y_te_raw >= np.percentile(y_te_raw, 90)],
            pred[y_te_raw >= np.percentile(y_te_raw, 90)]
        ),
        "mae_log_top99": mean_absolute_error(
            y_te[y_te_raw >= np.percentile(y_te_raw, 99)],
            pred[y_te_raw >= np.percentile(y_te_raw, 99)]
        ),
        "pinball_90": pinball_loss(y_te, pred, 0.9),
        "pinball_95": pinball_loss(y_te, pred, 0.95),
    }


In [2]:
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", "passthrough", FEATURE_COLS),
        ]
    )

def make_xgb():
    return XGBRegressor(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",       # was "hist"
        device="cuda",                # xgboost>=2.0, or use `gpu_id=0` on older releases
        random_state=0,
        n_jobs=0,                     # GPU path ignores CPU threads
    )

In [3]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import json
import pandas as pd

def downstream_eval_xgb(real_df, synth_df, seed=0):
    real_train, real_test = train_test_split(
        real_df, test_size=0.2, random_state=seed
    )

    # Target
    y_tr = np.log1p(real_train[TARGET].values)
    y_te = np.log1p(real_test[TARGET].values)
    y_te_raw = real_test[TARGET].values
    y_syn = np.log1p(synth_df[TARGET].values)

    # Drop columns
    real_train = real_train.drop(columns=DROP_COLS)
    real_test = real_test.drop(columns=DROP_COLS)
    synth_df = synth_df.drop(columns=DROP_COLS)
    
    # Preprocess (fit on real-train ONLY)
    pre = make_preprocessor()
    X_tr = pre.fit_transform(real_train)
    X_te = pre.transform(real_test)
    X_syn = pre.transform(synth_df)

    model = make_xgb()

    rr = evaluate_model(model, X_tr, y_tr, X_te, y_te, y_te_raw)
    sr = evaluate_model(model, X_syn, y_syn, X_te, y_te, y_te_raw)

    return {
        "RR": rr,
        "SR": sr,
        "Delta": {k: sr[k] - rr[k] for k in rr},
    }


In [4]:
real = pd.read_csv("../datasets/filtered.csv", engine="pyarrow")
synth_df = pd.read_csv("../output/before.csv", engine="pyarrow")

k = 3  # number of sigmas

# Compute statistics
mu = real[TARGET].mean()
sigma = real[TARGET].std()

# Filter DataFrame
real_filtered = real[(real[TARGET] >= mu - k * sigma) &
                 (real
                [TARGET] <= mu + k * sigma)]
synth_filtered = synth_df[(synth_df[TARGET] >= mu - k * sigma) &
                 (synth_df
                [TARGET] <= mu + k * sigma)]

results = downstream_eval_xgb(real_filtered, synth_filtered)

output_path = Path("../output/before.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
json_payload = json.dumps(results, indent=2, default=float)
output_path.write_text(json_payload)
print(json_payload)
print(f"Saved downstream metrics to {output_path}")


/home/seongho/gmm-synth-transfer/.venv/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [21:48:31] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


{
  "RR": {
    "mae_log": 0.6146430174999179,
    "spearman": 0.8943880423131255,
    "mae_log_top90": 1.3798786946731516,
    "mae_log_top99": 1.7841262717571382,
    "pinball_90": 0.3072891336957741,
    "pinball_95": 0.30728508681400096
  },
  "SR": {
    "mae_log": 0.8807472369742956,
    "spearman": 0.8369577432676824,
    "mae_log_top90": 1.8534719746265218,
    "mae_log_top99": 2.5940708374609946,
    "pinball_90": 0.42432839175532716,
    "pinball_95": 0.42232273841384954
  },
  "Delta": {
    "mae_log": 0.2661042194743777,
    "spearman": -0.05743029904544317,
    "mae_log_top90": 0.4735932799533702,
    "mae_log_top99": 0.8099445657038564,
    "pinball_90": 0.11703925805955306,
    "pinball_95": 0.11503765159984858
  }
}
Saved downstream metrics to ../output/before.json
